In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Point
from folium.plugins import TimestampedGeoJson, Timeline, TimelineSlider
from folium.features import GeoJson
from folium.utilities import JsCode
import folium
import plotly.express as px


In [ ]:
gdf = gpd.read_file("data/Ruskeasuo_2025-08-13.fgb")
# gdf = gpd.read_file("data/Kontula_2025-07-24.fgb")
print(gdf.head())

In [ ]:
gdf["center"] = gdf.geometry.centroid
gdf_tracks = gdf.sort_values(["id", "timestamp"])
gdf_tracks["timestamp_dt"] = pd.to_datetime(gdf_tracks["timestamp"], unit="ms")

def most_common(s):
    mode = s.mode()
    return mode.iloc[0] if len(mode) else None

def make_line(points):
    pts = [(p.x, p.y) for p in points if p.is_valid]
    return LineString(pts) if len(pts) > 1 else None

tracks = (
    gdf_tracks.groupby("id")
    .agg(
        start_timestamp=("timestamp","min"),
        end_timestamp=("timestamp","max"),
        start_timestamp_dt=("timestamp_dt","min"),
        end_timestamp_dt=("timestamp_dt","max"),
        point_count=("center","count"),
        predicted_class=("predicted_class",most_common),
        geometry=("center",make_line),
    )
).reset_index()

tracks_ok = tracks[tracks.geometry.notna()].copy()
broken = tracks[tracks.geometry.isna()].copy()

broken = broken.merge(gdf_tracks[["id","center"]], on="id")
broken["geometry"] = broken["center"]
broken = broken.drop(columns=["center"])

tracks_ok["track_type"] = "TRACK"
broken["track_type"] = "BROKEN_POINT"

tracks_all = pd.concat([tracks_ok, broken], ignore_index=True)
tracks_all = gpd.GeoDataFrame(tracks_all, geometry="geometry", crs=gdf_tracks.crs)

print("Tracks:", len(tracks_ok), " Broken:", len(broken))

PALETTE = ["red","blue","green","orange","purple"]
tracks_all["color"] = [PALETTE[i % 5] for i in range(len(tracks_all))]


def style_fn(feature):
    """Style for LineStrings + for Point border outline."""
    c = feature["properties"]["color"]
    return {
        "color": c,
        "weight": 1,
        "opacity": 0.5,
        "fill_color": c,
        "fill_opacity": 0.5,
    }

m = tracks_all.explore(
    style_kwds={"style_function": style_fn},
    marker_type="circle_marker",
    tooltip=["id", "track_type", "predicted_class", "point_count", "start_timestamp_dt", "end_timestamp_dt"],
    name="Tracks",
)

print(tracks_all.head())

m.save("tracks_and_broken.html")
print("Saved → tracks_and_broken.html")

In [ ]:
print(broken.head())

In [ ]:
print(tracks_all.head())

In [ ]:
tracks_all_wgs84 = tracks_all.copy()
if tracks_all.crs and tracks_all.crs.to_epsg() != 4326:
    tracks_all_wgs84 = tracks_all.to_crs(epsg=4326)

features = []
for _, row in tracks_all_wgs84.iterrows():
    geom = row.geometry
    if geom is None or not geom.is_valid:
        continue

    coordinates = []
    if geom.geom_type == "LineString":
        coordinates = [[coord[0], coord[1]] for coord in geom.coords]
    elif geom.geom_type == "Point":
        coordinates = [geom.x, geom.y]
    else:
        continue

    features.append({
        "type": "Feature",
        "geometry": {
            "type": geom.geom_type,
            "coordinates": coordinates,
        },
        "properties": {
            "id": row["id"],
            "track_type": row["track_type"],
            "predicted_class": row["predicted_class"],
            "point_count": row["point_count"],
            "start": row["start_timestamp"],
            "end": row["end_timestamp"],
            "start_timestamp_dt": row["start_timestamp_dt"].isoformat(),
            "end_timestamp_dt": row["end_timestamp_dt"].isoformat(),
            "color": row["color"],
        },
    })

tracks_geojson = {
    "type": "FeatureCollection",
    "features": features,
}


def style_fn(feature):
    return {
        "color": feature["properties"]["color"],
        "weight": 3,
        "opacity": 0.8,
    }

style_js = JsCode("""
    function (data) {
        return {
            color: data.properties.color,
            weight: 3,
            opacity: 0.8
        };
    }
""")

m = folium.Map(
    location=[tracks_all_wgs84.geometry.centroid.y.mean(),
              tracks_all_wgs84.geometry.centroid.x.mean()], 
    zoom_start=15,
)



timeline = Timeline(
    data=tracks_geojson,
    style=style_js,
).add_to(m)

tooltip = folium.GeoJsonTooltip(
    fields=["id", "track_type", "predicted_class", "point_count", "start_timestamp_dt", "end_timestamp_dt"],
    sticky=True,
)

tooltip.add_to(timeline)

TimelineSlider(
    auto_play=False,
    show_ticks=True,
    enable_keyboard_controls=True,
    playback_duration=30000,
).add_timelines(timeline).add_to(m) 

m.save("tracks_timeline_slider.html")
print("Saved → tracks_timeline_slider.html")

In [ ]:
print(len(tracks_geojson["features"]))

In [ ]:
print(tracks_geojson["features"][0])

In [ ]:
print(tracks_all_wgs84.head())

In [ ]:
tracks_ok_gdf = gpd.GeoDataFrame(tracks_ok, geometry="geometry", crs=gdf_tracks.crs)
tracks_ok_gdf = tracks_ok_gdf.to_crs(epsg=3067)

tracks_ok_gdf["length_m"] = tracks_ok_gdf.geometry.length

fig_len_vs_points = px.scatter(
    tracks_ok_gdf,
    x="point_count",
    y="length_m",
    color="predicted_class",
    opacity=0.3,
    labels={
        "point_count": "Number of points in track",
        "length_m": "Track length (m)",
        "predicted_class": "Predicted class",
    },
    title="Track length (m) vs. number of points",
)

fig_len_vs_points.update_layout(width=700, height=500)
fig_len_vs_points.show()

In [ ]:
speed_stats = (
    gdf_tracks.groupby("id")
    .agg(mean_speed=("speed", "mean"))
    .reset_index()
)

tracks_speed = tracks_ok.merge(speed_stats, on="id", how="left")

fig_speed_vs_points = px.scatter(
    tracks_speed,
    x="point_count",
    y="mean_speed",
    color="predicted_class",
    opacity=0.3,
    title="Mean speed vs. point count per track",
    labels={"mean_speed": "Mean speed (m/s)", "point_count": "Point count"},
)

fig_speed_vs_points.update_layout(width=700, height=500)
fig_speed_vs_points.show()

In [ ]:
classes = sorted(tracks_ok_gdf["predicted_class"].dropna().unique())

for cls in classes:
    df_cls = tracks_ok_gdf[tracks_ok_gdf["predicted_class"] == cls]

    fig_pc = px.histogram(
        df_cls,
        x="point_count",
        nbins=40,
        title=f"Point count distribution - class: {cls}",
        labels={"point_count": "Number of points in track"},
    )
    fig_pc.update_layout(
        width=650,
        height=400,
        bargap=0.05,
    )
    fig_pc.show()

In [ ]:
for cls in classes:
    df_cls = tracks_ok_gdf[tracks_ok_gdf["predicted_class"] == cls]

    fig_len = px.histogram(
        df_cls,
        x="length_m",
        nbins=40,
        title=f"Track length (m) distribution - class: {cls}",
        labels={"length_m": "Track length (m)"},
    )
    fig_len.update_layout(
        width=650,
        height=400,
        bargap=0.05,
    )
    fig_len.show()